# 05 — Tratamento dos NaNs

Passo 3 do roteiro de EDA (`docs/02_roteiro_eda.md`): decidir e documentar o que fazer com os meses sem dado — dropar, interpolar, ou deixar como está e tratar caso a caso na modelagem. O passo 4 (correlação) e o 5 (estacionariedade/Granger) do roteiro precisam de séries alinhadas e sem buraco, então essa decisão precisa estar tomada antes deles.

In [1]:
import pandas as pd
from IPython.display import display, Markdown

painel = pd.read_csv('https://raw.githubusercontent.com/vicleme/port-of-santos-credit-fx/refs/heads/main/data/processed/painel_2015_2025.csv')
painel['data'] = pd.to_datetime(painel['ano'].astype(str) + '-' + painel['mes'].astype(str) + '-01')
painel.isna().sum()

ano                            0
mes                            0
cambio_venda                   0
ativo_total                    3
credito_emprestimos_titulos    3
credito_financiamentos         3
credito_outras_operacoes       3
credito_total                  3
passivo_total                  3
exportação_fob                 0
importação_fob                 0
exportação_kg                  0
importação_kg                  0
data                           0
dtype: int64

## 1. Onde estão os NaNs

Só nas 6 colunas de origem ESTBAN (`ativo_total`, `credito_emprestimos_titulos`, `credito_financiamentos`, `credito_outras_operacoes`, `credito_total`, `passivo_total`) — `cambio_venda` e as 4 colunas de comércio exterior (Comex Stat) estão completas nos 132 meses. Isso já é esperado: o README documenta que a fonte ESTBAN, via Base dos Dados (plano gratuito), cobre só até 2025-09, enquanto câmbio e Comex Stat vão até 2025-12 (painel principal) ou além (painel estendido).

In [2]:
meses_com_nan = painel[painel.isna().any(axis=1)][['ano', 'mes']]
display(meses_com_nan)

primeiro_nan = meses_com_nan.index.min()
ultimo_nao_nan = primeiro_nan - 1
gap_no_meio = meses_com_nan.index.max() != len(painel) - 1

texto = (
    f"Os {len(meses_com_nan)} meses com NaN são exatamente os **3 últimos** do painel (out, nov, dez de 2025), "
    f"em linhas consecutivas até o fim da tabela (índices {meses_com_nan.index.min()} a {meses_com_nan.index.max()}, "
    f"de um total de {len(painel)} linhas). Ou seja: é uma **cauda censurada** (dado que ainda não foi reportado "
    "pela fonte), não um buraco no meio da série — distinção que importa pra decidir o tratamento a seguir."
)
display(Markdown(texto))

,ano,mes
129,2025,10
130,2025,11
131,2025,12


Os 3 meses com NaN são exatamente os **3 últimos** do painel (out, nov, dez de 2025), em linhas consecutivas até o fim da tabela (índices 129 a 131, de um total de 132 linhas). Ou seja: é uma **cauda censurada** (dado que ainda não foi reportado pela fonte), não um buraco no meio da série — distinção que importa pra decidir o tratamento a seguir.

## 2. Por que não interpolar

Interpolação (linear ou qualquer outro método) preenche um valor **entre** dois pontos conhecidos — pressupõe que existe um ponto depois do buraco pra ancorar a estimativa. Aqui não existe: os 3 meses faltantes são o final da série, então "interpolar" seria, na prática, **extrapolar** a partir da tendência recente — um exercício bem menos confiável, e ainda mais arriscado neste caso específico.

O notebook `03_agencias_ativo_credito.ipynb` já mostrou que `ativo_total`/`credito_total` são séries **estruturalmente voláteis** — altas de até +80% num ano, quedas de até -48% —, sem um padrão suave que uma extrapolação capturaria bem. Preencher out-dez/2025 com base nos meses anteriores daria uma falsa sensação de precisão sobre um trimestre que pode perfeitamente repetir uma dessas oscilações grandes.

In [3]:
variacao_2025 = painel[(painel['ano'] == 2025) & (painel['mes'] <= 9)]['credito_total'].pct_change().dropna()

texto = (
    f"Só pra ilustrar o risco: olhando só os meses de 2025 já conhecidos (jan-set), a variação mês a mês de "
    f"`credito_total` foi de **{variacao_2025.min()*100:.0f}%** a **+{variacao_2025.max()*100:.0f}%** — ou seja, "
    "mesmo no trecho imediatamente anterior ao buraco, a série continua mostrando saltos grandes de um mês pro "
    "outro. Não existe uma base recente estável o bastante pra ancorar uma extrapolação de out-dez/2025 com "
    "alguma confiança; um valor 'estimado' ali criaria uma ilusão de dado onde não há dado."
)
display(Markdown(texto))

Só pra ilustrar o risco: olhando só os meses de 2025 já conhecidos (jan-set), a variação mês a mês de `credito_total` foi de **-21%** a **+45%** — ou seja, mesmo no trecho imediatamente anterior ao buraco, a série continua mostrando saltos grandes de um mês pro outro. Não existe uma base recente estável o bastante pra ancorar uma extrapolação de out-dez/2025 com alguma confiança; um valor 'estimado' ali criaria uma ilusão de dado onde não há dado.

## 3. Decisão adotada

**Dropar os 3 meses (out-dez/2025), mas só para as análises que usam as colunas de crédito/balanço bancário.**

- Para o **passo 4 (correlação)** e o **passo 5 (estacionariedade / Granger)** envolvendo `credito_total` (ou qualquer uma das outras 5 colunas ESTBAN): usar o recorte **2015-01 a 2025-09** (129 dos 132 meses) — as séries de câmbio e comércio exterior perdem 3 meses reais de dado, mas isso garante que todas as séries usadas juntas em uma mesma análise multivariada estejam alinhadas, sem NaN.
- Para qualquer análise que **não** envolva as colunas ESTBAN (ex.: um estudo de sazonalidade só de `exportação_fob`), não há motivo pra descartar esses 3 meses — o painel completo (132 meses) continua válido, como já foi usado no `01` e no `04`.
- **Não interpolar/extrapolar** os 3 meses de crédito, pelos motivos da seção 2.
- **Não deixar como está** para as análises multivariadas — um NaN não tratado quebra ou distorce silenciosamente correlação, ADF e Granger, dependendo de como cada função/biblioteca lida com valor ausente (algumas descartam a linha automaticamente, outras geram erro, outras simplesmente calculam errado sem avisar). Preferível tornar o recorte explícito.

Isso é consistente com o próprio roteiro, que já lista essa cautela como obrigatória antes do passo 4: séries alinhadas e sem buraco silencioso.

In [4]:
# painel_credito: recorte usado a partir daqui em análises que envolvem as colunas ESTBAN
# (correlação, estacionariedade/Granger com credito_total e afins)
painel_credito = painel[painel['data'] <= '2025-09-01'].copy()

print(f"painel completo: {len(painel)} meses (2015-01 a {painel['data'].max().strftime('%Y-%m')})")
print(f"painel_credito:  {len(painel_credito)} meses (2015-01 a {painel_credito['data'].max().strftime('%Y-%m')})")
print(f"NaNs restantes em painel_credito: {painel_credito.isna().sum().sum()}")

painel completo: 132 meses (2015-01 a 2025-12)
painel_credito:  129 meses (2015-01 a 2025-09)
NaNs restantes em painel_credito: 0


## Resumo do passo 3

- NaNs existem só nas 6 colunas ESTBAN, concentrados nos 3 últimos meses do painel (out-dez/2025) — cauda censurada por defasagem de reporte da fonte gratuita, não buraco no meio da série.
- Câmbio e comércio exterior (Comex Stat) estão completos nos 132 meses e não precisam de nenhum tratamento.
- Decisão: **dropar** os 3 meses finais para qualquer análise multivariada que envolva colunas ESTBAN, usando o recorte `painel_credito` (2015-01 a 2025-09, 129 meses). Interpolação/extrapolação foi descartada por ser uma cauda sem ponto de ancoragem futuro, numa série já comprovadamente volátil (notebook 03).
- Análises que não tocam nas colunas ESTBAN continuam usando o painel completo de 132 meses, sem necessidade de recorte.

**Próximo passo (04 do roteiro):** correlação — matriz de correlação entre câmbio, `credito_total`, `exportação_fob` e `importação_fob`, usando `painel_credito`.